In [5]:
# Open the new type of netcdf file with parameters and spectra
# A. Lefauve, 2026

import h5py
path = "/lustre/orion/cfd135/proj-shared/Hsst/R1P7/001_Final/R1P7.nc"
f = h5py.File(path, "r")

print("Top-level keys:", list(f.keys()))
print("Top-level keys:", list(f.keys()))

def walk(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(name, obj.shape, obj.dtype)

f.visititems(walk)

Top-level keys: ['ke_x', 'kx', 'ky', 'ke_y', 'kz', 'ke_z', 'k3', 'ke_3', 'pe_x', 'pe_y', 'pe_z', 'pe_3', 'ek_x', 'ek_y', 'ek_z', 'ek_3', 'ep_x', 'ep_y', 'ep_z', 'ep_3', '1', 'nu', 'D', 'Pr', 'g_rho0', 't', 'Lk', 'Lb', 'Fr', 'Ri', 'Gn', 'Res', 'Re', 'Ss', 'Ns', 'LambdaS', 'Gamma1', 'Gamma2', 'kmaxLb', 'Nx', 'Ek', 'pb', 'ps', 'ek', 'Ep', 'ep']
Top-level keys: ['ke_x', 'kx', 'ky', 'ke_y', 'kz', 'ke_z', 'k3', 'ke_3', 'pe_x', 'pe_y', 'pe_z', 'pe_3', 'ek_x', 'ek_y', 'ek_z', 'ek_3', 'ep_x', 'ep_y', 'ep_z', 'ep_3', '1', 'nu', 'D', 'Pr', 'g_rho0', 't', 'Lk', 'Lb', 'Fr', 'Ri', 'Gn', 'Res', 'Re', 'Ss', 'Ns', 'LambdaS', 'Gamma1', 'Gamma2', 'kmaxLb', 'Nx', 'Ek', 'pb', 'ps', 'ek', 'Ep', 'ep']
1 (1,) >f4
D (1,) float64
Ek (1,) float64
Ep (1,) float64
Fr (1,) float64
Gamma1 (1,) float64
Gamma2 (1,) float64
Gn (1,) float64
LambdaS (1,) float64
Lb (1,) float64
Lk (1,) float64
Ns (1,) float64
Nx (1,) float64
Pr (1,) float64
Re (1,) float64
Res (1,) float64
Ri (1,) float64
Ss (1,) float64
ek (1,) float64


In [2]:
# Export to Excel
import h5py
import pandas as pd
import os
!pip install openpyxl
path = "/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/R1P1.nc"

# extract just the filename (R1P1.nc → R1P1 → R1P1.xlsx)
filename = os.path.basename(path)            # R1P1.nc
name = os.path.splitext(filename)[0]         # R1P1
output_path = os.path.join(os.path.dirname(path), name + ".xlsx")

print(output_path)  # sanity check

with h5py.File(path, "r") as f:
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for key in f.keys():
            data = f[key][()].flatten()
            df = pd.DataFrame({key: data})
            df.to_excel(writer, sheet_name=key[:31], index=False)

print(f"Saved to {output_path}")

/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/R1P1.xlsx
Saved to /lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/R1P1.xlsx


In [13]:
# Now group all .nc in one directory and export xlsx

import h5py
import pandas as pd
import os
import shutil

folders = ["R1P1", "R1P7", "R1P50", "R4P1", "R4P7", "R4P50", "R6P1", "R6P7", "R6P50","R8P1", "R8P7","R10P1", "R10P7" ]  # your list
base_path = "/lustre/orion/cfd135/proj-shared/Hsst"

# destination folder
dest_path = "/lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS"
os.makedirs(dest_path, exist_ok=True)

for folder in folders:
    source_folder = os.path.join(base_path, folder, "001_Final")

    if not os.path.exists(source_folder):
        print(f"Skipping {folder}: path not found")
        continue

    for file in os.listdir(source_folder):
        if file.endswith(".nc"):

            source_file = os.path.join(source_folder, file)
            dest_file = os.path.join(dest_path, file)

            print(f"\nProcessing {source_file}")

            # copy the .nc file
            shutil.copy2(source_file, dest_file)

            # output Excel file in destination folder
            name = os.path.splitext(file)[0]
            output_path = os.path.join(dest_path, name + ".xlsx")

            try:
                with h5py.File(source_file, "r") as f:
                    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

                        # --- first sheet: all scalar parameters ---
                        param_names = []
                        param_values = []

                        for key in f.keys():
                            data = f[key][()]
                            if data.size == 1:
                                param_names.append(key)
                                param_values.append(data.item())

                        params_df = pd.DataFrame({
                            "parameter": param_names,
                            "value": param_values
                        })
                        params_df.to_excel(writer, sheet_name="parameters", index=False)

                        # --- other sheets: non-scalar datasets only ---
                        for key in f.keys():
                            data = f[key][()]
                            if data.size == 1:
                                continue

                            data = data.flatten()
                            sheet_name = key[:31].replace("/", "_")
                            df = pd.DataFrame({key: data})
                            df.to_excel(writer, sheet_name=sheet_name, index=False)

                print(f"Saved → {output_path}")

            except Exception as e:
                print(f"Failed on {source_file}: {e}")


Processing /lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/R1P1.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R1P1.xlsx

Processing /lustre/orion/cfd135/proj-shared/Hsst/R1P7/001_Final/R1P7.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R1P7.xlsx

Processing /lustre/orion/cfd135/proj-shared/Hsst/R1P50/001_Final/R1P50.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R1P50.xlsx

Processing /lustre/orion/cfd135/proj-shared/Hsst/R4P1/001_Final/R4P1.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R4P1.xlsx

Processing /lustre/orion/cfd135/proj-shared/Hsst/R4P7/001_Final/R4P7.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R4P7.xlsx

Processing /lustre/orion/cfd135/proj-shared/Hsst/R4P50/001_Final/R4P50.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R4P50.xlsx

Processing /lustre/orion/cfd135/proj-shared/Hsst/R6P1/001_Final/R6P1.nc
Saved → /lustre/orion/cfd135/proj-shared/Hsst/ALL_OUTPUTS/R6P1.xlsx

Proces

In [1]:
# NO LONGER NEEDED: New code to open the new type of netcdf file sent by Steve mid January 2025
import h5py

path = "/lustre/orion/cfd135/proj-shared/Hsst/R1P1/SpectraAdrien/Stats/R1P1.nc"
f = h5py.File(path, "r")

print("Top-level keys:", list(f.keys()))

def walk(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(name, obj.shape, obj.dtype)

f.visititems(walk)

Top-level keys: ['ke_x', 'kx', 'ky', 'ke_y', 'kz', 'ke_z', 'pe_x', 'pe_y', 'pe_z', 'ek_x', 'ek_y', 'ek_z', 'ep_x', 'ep_y', 'ep_z', '1', 'nu', 'D', 'Pr', 'g_rho0', 'Lk', 'Lb', 'Fr', 'Ri', 'Gn', 'Res', 'Re', 'Ss', 'Ns', 'LambdaS', 'Gamma1', 'Gamma2', 'kmaxLb', 'Nx', 'Ek', 'pb', 'ps', 'ek', 'Ep', 'ep']
1 (1,) >f4
D (1,) float64
Ek (1,) float64
Ep (1,) float64
Fr (1,) float64
Gamma1 (1,) float64
Gamma2 (1,) float64
Gn (1,) float64
LambdaS (1,) float64
Lb (1,) float64
Lk (1,) float64
Ns (1,) float64
Nx (1,) float64
Pr (1,) float64
Re (1,) float64
Res (1,) float64
Ri (1,) float64
Ss (1,) float64
ek (1,) float64
ek_x (769,) float64
ek_y (385,) float64
ek_z (193,) float64
ep (1,) float64
ep_x (769,) float64
ep_y (385,) float64
ep_z (193,) float64
g_rho0 (1,) float64
ke_x (769,) float64
ke_y (385,) float64
ke_z (193,) float64
kmaxLb (1,) float64
kx (769,) float32
ky (385,) float32
kz (193,) float32
nu (1,) float64
pb (1,) float64
pe_x (769,) float64
pe_y (385,) float64
pe_z (193,) float64
ps (1

In [2]:
# NO LONGER NEEDED: New code to inspect the new type of netcdf file created by Steve late January 2025 
from scipy.io import netcdf_file

path = "/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/Spectra/mespec_1100.000000.nc"
nc = netcdf_file(path, "r")

print("Dimensions:")
for k, n in nc.dimensions.items():
    print(f"  {k:20s} {n}")

print("\nVariables:")
for k, v in nc.variables.items():
    # v.typecode() is the scipy netcdf dtype indicator (like 'f', 'd', 'i')
    print(f"  {k:20s} shape={v.shape} typecode={v.typecode()} dims={v.dimensions}")

print("\nGlobal attributes:")
for a in nc._attributes:
    print(f"  {a} = {getattr(nc, a)}")

Dimensions:
  nkx                  769
  nky                  385
  nkz                  193
  nks                  193
  1                    1

Variables:
  kx                   shape=(769,) typecode=f dims=('nkx',)
  ky                   shape=(385,) typecode=f dims=('nky',)
  kz                   shape=(193,) typecode=f dims=('nkz',)
  k3                   shape=(193,) typecode=f dims=('nks',)
  time                 shape=(1,) typecode=f dims=('1',)
  ke_x                 shape=(769,) typecode=f dims=('nkx',)
  ke_y                 shape=(385,) typecode=f dims=('nky',)
  ke_z                 shape=(193,) typecode=f dims=('nkz',)
  ke_3                 shape=(193,) typecode=f dims=('nks',)
  pb_x                 shape=(769,) typecode=f dims=('nkx',)
  pb_y                 shape=(385,) typecode=f dims=('nky',)
  pb_z                 shape=(193,) typecode=f dims=('nkz',)
  pb_3                 shape=(193,) typecode=f dims=('nks',)
  ps_x                 shape=(769,) typecode=f dims=('